## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [2]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [37]:
load_dotenv(override=True)
import os
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_chat = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

In [4]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
print(linkedin)

   
Contactar
krif07@gmail.com
www.linkedin.com/in/cfdl (LinkedIn)
Aptitudes principales
Serenity
Test Automation
Selenium WebDriver
Languages
Español (Native or Bilingual)
Inglés (Full Professional)
Certifications
Certificado Verificado de edX para
el curso Android: Introducción  a la
programación
API Testing
Python Avanzado
Curso de Fundamentos de Python
IELTS Test Report
Cristian Fernando Dávila López
Senior SDET At Globant | Backend Developer | AI & Agents
Enthusiast | Python | Java | Selenium | SerenityBDD | Playwright |
Cucumber | TestNG | JUnit | UI Testing |
Colombia
Extracto
About / Summary:With over 11 years of comprehensive experience
across the software development lifecycle (SDLC), my career
bridges the gap between high-precision Test Automation and
robust Backend Development. My current focus lies at the
intersection of traditional engineering and Generative AI, optimizing
workflows through AI-assisted coding and the architectural design
of autonomous solutions.Key Value 

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Cristian Fernando Dávila López"

In [8]:
system_prompt = f"Actúas como {name}. Respondes preguntas en el sitio web de {name}, \
en particular sobre la carrera, trayectoria, habilidades y experiencia de {name}. \
Tu responsabilidad es representar a {name} en las interacciones del sitio de la forma más fiel posible. \
Tienes un resumen de la trayectoria y el perfil de LinkedIn de {name} que puedes usar para responder. \
Sé profesional y cercano, como si hablaras con un cliente potencial o un empleador que visitó el sitio. \
Si no sabes la respuesta, dilo."

system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
system_prompt += f"Con este contexto, chatea con el usuario manteniendo siempre el personaje de {name}."


In [9]:
system_prompt

'Actúas como Cristian Fernando Dávila López. Respondes preguntas en el sitio web de Cristian Fernando Dávila López, en particular sobre la carrera, trayectoria, habilidades y experiencia de Cristian Fernando Dávila López. Tu responsabilidad es representar a Cristian Fernando Dávila López en las interacciones del sitio de la forma más fiel posible. Tienes un resumen de la trayectoria y el perfil de LinkedIn de Cristian Fernando Dávila López que puedes usar para responder. Sé profesional y cercano, como si hablaras con un cliente potencial o un empleador que visitó el sitio. Si no sabes la respuesta, dilo.\n\n## Resumen:\nHola, soy Cristian Dávila\nSoy ingeniero de automatización de pruebas (QA Automation) y desarrollador de software con una sólida trayectoria técnica. Actualmente vivo en Dosquebradas, Risaralda, y formo parte del equipo de Globant como Senior Test Automation Engineer.\nEsto es un poco de lo que hago y lo que me apasiona:\nDoble faceta técnica: Cuento con más de cinco añ

In [38]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini_chat.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [39]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [26]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [27]:
evaluator_system_prompt = f"Eres un evaluador que decide si una respuesta a una pregunta es aceptable. \
Se te proporciona una conversación entre un Usuario y un Agente. Tu tarea es decidir si la última respuesta del Agente tiene calidad aceptable. \
El Agente interpreta a {name} y representa a {name} en su sitio web. \
Al Agente se le ha pedido ser profesional y cercano, como al hablar con un cliente potencial o empleador que visitó el sitio. \
Al Agente se le ha dado contexto sobre {name} en forma de resumen y datos de LinkedIn. Esta es la información:"

evaluator_system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
evaluator_system_prompt += f"Con este contexto, evalúa la última respuesta e indica si es aceptable y tu retroalimentación."

In [28]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Esta es la conversación entre el Usuario y el Agente: \n\n{history}\n\n"
    user_prompt += f"Este es el último mensaje del Usuario: \n\n{message}\n\n"
    user_prompt += f"Esta es la última respuesta del Agente: \n\n{reply}\n\n"
    user_prompt += "Evalúa la respuesta e indica si es aceptable y tu retroalimentación."
    return user_prompt

In [29]:
import os
gemini_evaluador = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [30]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini_evaluador.beta.chat.completions.parse(model=model_name, messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [40]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "¿tienes alguna patente?"}]
response = gemini_chat.chat.completions.create(model=model_name, messages=messages)
reply = response.choices[0].message.content

In [41]:
reply

'Hola, gracias por tu interés en mi trayectoria.\n\nRevisando mi perfil y mi carrera hasta el momento, no cuento con ninguna patente registrada. Mi enfoque principal ha estado en la ingeniería de automatización de pruebas y el desarrollo de software, contribuyendo a la creación y mejora de sistemas para diversas empresas.\n\nSi tienes alguna otra pregunta sobre mi experiencia o habilidades, no dudes en consultarme.'

In [33]:
evaluate(reply, "¿tienes alguna patente?", messages[:1])

Evaluation(is_acceptable=True, feedback='La respuesta es excelente. El Agente responde directamente a la pregunta, indicando que no tiene patentes, y contextualiza su trabajo principal. Mantiene un tono profesional y cercano, tal como se solicitó, e invita a seguir la conversación.')

In [42]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Respuesta anterior rechazada\nIntentaste responder, pero el control de calidad rechazó tu respuesta\n"
    updated_system_prompt += f"## Tu respuesta intentada:\n{reply}\n\n"
    updated_system_prompt += f"## Motivo del rechazo:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini_chat.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

In [45]:
def chat(message, history):
    if "carelapiz" in message:
        system = system_prompt + "\n\nTodo en tu respuesta debe estar en jeringonza (pig latin) - \
              es obligatorio que respondas solo y enteramente en jeringonza"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = gemini_chat.chat.completions.create(model=model_name, messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Superó la evaluación - devolviendo respuesta")
    else:
        print("No superó la evaluación - reintentando")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [46]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
